# Function 1: Data Collection & Initial Dataset Profiling (Olist E-Commerce)

This notebook loads the 9 raw Olist datasets from `raw_data/`, profiles row/column dimensions, checks data types, missing values, duplicates, primary/composite keys, and foreign-key referential integrity, and exports initial profiling reports to `profiling_reports/`.

In [12]:
from pathlib import Path
import zipfile
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'raw_data'
OUT_DIR = BASE_DIR / 'profiling_reports'
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# Extract ZIP if raw CSVs are zipped
zip_candidates = list(BASE_DIR.glob('*.zip')) + list(DATA_DIR.glob('*.zip'))
if zip_candidates and not list(DATA_DIR.glob('*.csv')):
    zip_path = zip_candidates[0]
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print(f'Extracted: {zip_path.name} -> {DATA_DIR}')

csv_files = sorted(
    path for path in DATA_DIR.glob('*.csv')
    if not path.name.startswith('dataset_')
)

if not csv_files:
    raise FileNotFoundError(
        'No original Olist CSV files found in the raw_data folder.'
    )

print(f'Found {len(csv_files)} CSV files in {DATA_DIR}:')
for path in csv_files:
    print('-', path.name)


Found 9 CSV files:
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


## 1. تحميل الملفات وعرض حجم كل ملف
هنسجل عدد الصفوف والأعمدة لكل ملف، من غير تعديل البيانات الأصلية.

In [ ]:
dataframes = {}
load_errors = []

for path in csv_files:
    try:
        
        df = pd.read_csv(path, low_memory=False)
        dataframes[path.stem] = df
        print(f'{path.name}: {df.shape[0]:,} rows × {df.shape[1]} columns')
    except Exception as exc:
        load_errors.append({'file': path.name, 'error': str(exc)})
        print(f'Could not load {path.name}: {exc}')

if load_errors:
    display(pd.DataFrame(load_errors))


olist_customers_dataset.csv: 99,441 rows × 5 columns
olist_geolocation_dataset.csv: 1,000,163 rows × 5 columns
olist_order_items_dataset.csv: 112,650 rows × 7 columns
olist_order_payments_dataset.csv: 103,886 rows × 5 columns
olist_order_reviews_dataset.csv: 99,224 rows × 7 columns
olist_orders_dataset.csv: 99,441 rows × 8 columns
olist_products_dataset.csv: 32,951 rows × 9 columns
olist_sellers_dataset.csv: 3,095 rows × 4 columns
product_category_name_translation.csv: 71 rows × 2 columns


## 2. ملخص حجم الملفات
الملخص ده مفيد للتوثيق ومقارنة عدد الصفوف بعد تحميل البيانات إلى SQLite.

In [14]:
size_rows = []
for name, df in dataframes.items():
    size_rows.append({
        'table_candidate': name.replace('_dataset', ''),
        'source_file': name + '.csv',
        'rows': len(df),
        'columns': len(df.columns),
        'memory_mb_approx': round(df.memory_usage(deep=True).sum() / (1024**2), 2)
    })

file_summary = pd.DataFrame(size_rows).sort_values('source_file').reset_index(drop=True)
display(file_summary)
file_summary.to_csv(OUT_DIR / 'dataset_file_summary.csv', index=False)
print('Saved: profiling_reports/dataset_file_summary.csv')


,table_candidate,source_file,rows,columns,memory_mb_approx
0,olist_customers,olist_customers_dataset.csv,99441,5,11.03
1,olist_geolocation,olist_geolocation_dataset.csv,1000163,5,50.12
2,olist_order_items,olist_order_items_dataset.csv,112650,7,18.37
3,olist_order_payments,olist_order_payments_dataset.csv,103886,5,8.11
4,olist_order_reviews,olist_order_reviews_dataset.csv,99224,7,17.84
5,olist_orders,olist_orders_dataset.csv,99441,8,21.95
6,olist_products,olist_products_dataset.csv,32951,9,3.73
7,olist_sellers,olist_sellers_dataset.csv,3095,4,0.22
8,product_category_name_translation,product_category_name_translation.csv,71,2,0.00


Saved: dataset_file_summary.csv


## 3. أسماء الأعمدة وأنواع البيانات
هنراجع أسماء الأعمدة والـ dtypes؛ أنواع Pandas دي مؤشرات أولية وليست بالضرورة أنواع SQLite النهائية.

In [15]:
column_rows = []
for name, df in dataframes.items():
    for col in df.columns:
        column_rows.append({
            'table_candidate': name.replace('_dataset', ''),
            'column': col,
            'pandas_dtype': str(df[col].dtype),
            'non_null_count': int(df[col].notna().sum()),
            'missing_count': int(df[col].isna().sum()),
            'missing_pct': round(df[col].isna().mean() * 100, 2),
            'distinct_non_null': int(df[col].nunique(dropna=True))
        })

column_profile = pd.DataFrame(column_rows)
display(column_profile)
column_profile.to_csv(OUT_DIR / 'dataset_column_profile.csv', index=False)
print('Saved: profiling_reports/dataset_column_profile.csv')


,table_candidate,column,pandas_dtype,non_null_count,missing_count,missing_pct,distinct_non_null
0,olist_customers,customer_id,str,99441,0,0.00,99441
1,olist_customers,customer_unique_id,str,99441,0,0.00,96096
2,olist_customers,customer_zip_code_prefix,int64,99441,0,0.00,14994
3,olist_customers,customer_city,str,99441,0,0.00,4119
4,olist_customers,customer_state,str,99441,0,0.00,27
5,olist_geolocation,geolocation_zip_code_prefix,int64,1000163,0,0.00,19015
6,olist_geolocation,geolocation_lat,float64,1000163,0,0.00,717360
7,olist_geolocation,geolocation_lng,float64,1000163,0,0.00,717613
8,olist_geolocation,geolocation_city,str,1000163,0,0.00,8011
9,olist_geolocation,geolocation_state,str,1000163,0,0.00,27


Saved: dataset_column_profile.csv


## 4. القيم المفقودة والتكرارات الكاملة
ده فحص أولي فقط. التكرار الكامل يختلف عن تكرار قيمة المفتاح، وهنراجع كل حالة قبل اتخاذ قرار.

In [ ]:
quality_rows = []
for name, df in dataframes.items():
    quality_rows.append({
        'table_candidate': name.replace('_dataset', ''),
        'rows': len(df),
        'full_duplicate_rows': int(df.duplicated().sum()),
        'cells_missing': int(df.isna().sum().sum()),
        'columns_with_missing_values': int(df.isna().any().sum())
    })

quality_summary = pd.DataFrame(quality_rows).sort_values('table_candidate').reset_index(drop=True)
display(quality_summary)
quality_summary.to_csv(OUT_DIR / 'dataset_initial_quality_checks.csv', index=False)
print('Saved: profiling_reports/dataset_initial_quality_checks.csv')


display(column_profile[column_profile['missing_count'] > 0]
        .sort_values(['table_candidate', 'missing_pct'], ascending=[True, False]))


,table_candidate,rows,full_duplicate_rows,cells_missing,columns_with_missing_values
0,olist_customers,99441,0,0,0
1,olist_geolocation,1000163,261831,0,0
2,olist_order_items,112650,0,0,0
3,olist_order_payments,103886,0,0,0
4,olist_order_reviews,99224,0,145903,2
5,olist_orders,99441,0,4908,3
6,olist_products,32951,0,2448,8
7,olist_sellers,3095,0,0,0
8,product_category_name_translation,71,0,0,0


Saved: dataset_initial_quality_checks.csv


,table_candidate,column,pandas_dtype,non_null_count,missing_count,missing_pct,distinct_non_null
25,olist_order_reviews,review_comment_title,str,11568,87656,88.34,4527
26,olist_order_reviews,review_comment_message,str,40977,58247,58.70,36159
35,olist_orders,order_delivered_customer_date,str,96476,2965,2.98,95664
34,olist_orders,order_delivered_carrier_date,str,97658,1783,1.79,81018
33,olist_orders,order_approved_at,str,99281,160,0.16,90733
38,olist_products,product_category_name,str,32341,610,1.85,73
39,olist_products,product_name_lenght,float64,32341,610,1.85,66
40,olist_products,product_description_lenght,float64,32341,610,1.85,2960
41,olist_products,product_photos_qty,float64,32341,610,1.85,19
42,olist_products,product_weight_g,float64,32949,2,0.01,2204


## 5. فحص المفاتيح المحتملة


In [17]:
candidate_keys = {
    'olist_customers_dataset': ['customer_id', 'customer_unique_id'],
    'olist_orders_dataset': ['order_id', 'customer_id'],
    'olist_order_items_dataset': ['order_id', 'order_item_id', 'product_id', 'seller_id'],
    'olist_order_payments_dataset': ['order_id', 'payment_sequential'],
    'olist_order_reviews_dataset': ['review_id', 'order_id'],
    'olist_products_dataset': ['product_id'],
    'olist_sellers_dataset': ['seller_id'],
    'product_category_name_translation': ['product_category_name'],
    'olist_geolocation_dataset': []
}

key_rows = []
for table_name, candidates in candidate_keys.items():
    df = dataframes.get(table_name)
    if df is None:
        continue
    for col in candidates:
        if col in df.columns:
            key_rows.append({
                'table': table_name,
                'column_candidate': col,
                'null_values': int(df[col].isna().sum()),
                'distinct_values': int(df[col].nunique(dropna=True)),
                'duplicate_non_null_rows': int(df[col].dropna().duplicated().sum()),
                'unique_non_null': bool(df[col].dropna().is_unique)
            })

key_profile = pd.DataFrame(key_rows)
display(key_profile)
key_profile.to_csv(OUT_DIR / 'dataset_key_candidates.csv', index=False)
print('Saved: profiling_reports/dataset_key_candidates.csv')


,table,column_candidate,null_values,distinct_values,duplicate_non_null_rows,unique_non_null
0,olist_customers_dataset,customer_id,0,99441,0,True
1,olist_customers_dataset,customer_unique_id,0,96096,3345,False
2,olist_orders_dataset,order_id,0,99441,0,True
3,olist_orders_dataset,customer_id,0,99441,0,True
4,olist_order_items_dataset,order_id,0,98666,13984,False
5,olist_order_items_dataset,order_item_id,0,21,112629,False
6,olist_order_items_dataset,product_id,0,32951,79699,False
7,olist_order_items_dataset,seller_id,0,3095,109555,False
8,olist_order_payments_dataset,order_id,0,99440,4446,False
9,olist_order_payments_dataset,payment_sequential,0,29,103857,False


Saved: dataset_key_candidates.csv


## 6. Output Profiling Reports
All profiling summaries are saved inside `profiling_reports/`:
- `profiling_reports/dataset_file_summary.csv`
- `profiling_reports/dataset_column_profile.csv`
- `profiling_reports/dataset_initial_quality_checks.csv`
- `profiling_reports/dataset_key_candidates.csv`


In [18]:

# 1. Check whether composite keys are unique

composite_keys = {
    "olist_order_items_dataset": [
        "order_id", "order_item_id"
    ],
    "olist_order_payments_dataset": [
        "order_id", "payment_sequential"
    ],
}

for table_name, columns in composite_keys.items():
    df = dataframes[table_name]

    duplicate_count = df.duplicated(
        subset=columns, keep=False
    ).sum()

    print(f"\nTable: {table_name}")
    print(f"Candidate key: {columns}")
    print(f"Duplicate rows in key: {duplicate_count}")
    print(f"Unique key: {duplicate_count == 0}")


# 2. Check foreign-key references

foreign_keys = [
    ("olist_orders_dataset", "customer_id",
     "olist_customers_dataset", "customer_id"),

    ("olist_order_items_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("olist_order_items_dataset", "product_id",
     "olist_products_dataset", "product_id"),

    ("olist_order_items_dataset", "seller_id",
     "olist_sellers_dataset", "seller_id"),

    ("olist_order_payments_dataset", "order_id",
     "olist_orders_dataset", "order_id"),

    ("olist_order_reviews_dataset", "order_id",
     "olist_orders_dataset", "order_id"),
]

for child_table, child_col, parent_table, parent_col in foreign_keys:
    child = dataframes[child_table]
    parent = dataframes[parent_table]

    parent_values = set(parent[parent_col].dropna())

    orphan_count = (
        child[child_col].notna()
        & ~child[child_col].isin(parent_values)
    ).sum()

    print(
        f"{child_table}.{child_col} -> "
        f"{parent_table}.{parent_col}: "
        f"{orphan_count} unmatched rows"
    )



Table: olist_order_items_dataset
Candidate key: ['order_id', 'order_item_id']
Duplicate rows in key: 0
Unique key: True

Table: olist_order_payments_dataset
Candidate key: ['order_id', 'payment_sequential']
Duplicate rows in key: 0
Unique key: True
olist_orders_dataset.customer_id -> olist_customers_dataset.customer_id: 0 unmatched rows
olist_order_items_dataset.order_id -> olist_orders_dataset.order_id: 0 unmatched rows
olist_order_items_dataset.product_id -> olist_products_dataset.product_id: 0 unmatched rows
olist_order_items_dataset.seller_id -> olist_sellers_dataset.seller_id: 0 unmatched rows
olist_order_payments_dataset.order_id -> olist_orders_dataset.order_id: 0 unmatched rows
olist_order_reviews_dataset.order_id -> olist_orders_dataset.order_id: 0 unmatched rows


In [19]:
# Check whether every order references an existing customer

customer_ids = set(dataframes["olist_customers_dataset"]["customer_id"])

orders_without_customer = dataframes["olist_orders_dataset"][
    ~dataframes["olist_orders_dataset"]["customer_id"].isin(customer_ids)
]

print("Orders without a matching customer:", len(orders_without_customer))

Orders without a matching customer: 0
